# 03 · Autograd and the training loop

In Lesson 2, a least-squares solver fitted a line in one call. This time we'll fit the same kind of line by taking small steps. We'll reuse the synthetic discharge series and start with a single forecast we can work out on paper. Then we'll run the same operations through two Python files that the notebook imports.

## 1 · What the loss measures

Use day 10 flow (4 m³/s) to predict day 11 flow (observed at 3 m³/s). Start with a **weight** of 1 and a **bias** of 0. A weight multiplies an input, and a bias adds a constant. In this model they are the slope and intercept.

The **forward pass** applies the model to an input. Here, the prediction is 1 × 4 + 0 = 4 m³/s. The **residual** in this lesson is prediction minus observation, so it is +1 m³/s. Squaring gives a loss of 1. For a batch, **mean squared error (MSE)** squares each residual and then averages. A forecast that misses by 2 contributes 4 to that average.

**Try it.** If the predicted flow were 2.5, what would the residual and squared error be?

<details>
<summary>Show solution</summary>

The residual is −0.5 m³/s and the squared error is 0.25 (m³/s)².

</details>

## 2 · Ask which way the loss changes

A **gradient** gives the local rate of change of the loss with respect to each parameter. At our starting point, increasing the bias by a tiny amount increases the prediction by that same amount. Increasing the weight by that amount increases the prediction four times as much, because the input is 4.

For squared error, the derivative with respect to the prediction is twice the residual. This is 2 × 1 = 2. To get the derivative with respect to the weight, multiply that by how much the prediction changes per unit weight, which is 4. This is the **chain rule**. The weight gradient is 8 and the bias gradient is 2.

PyTorch **autograd** records how tensors were combined. Calling `backward()` applies the chain rule back through those operations and stores the gradients in `.grad`. It leaves the parameter values as they were.

In [5]:
import torch  # Provides tensors and automatic differentiation.

# These scalar tensors have shape (). Record operations involving weight
# and bias so we can calculate how the loss changes when they change.
weight = torch.tensor(1.0, requires_grad=True)
bias = torch.tensor(0.0, requires_grad=True)
today = torch.tensor(4.0)     # Day 10 input, held fixed.
tomorrow = torch.tensor(3.0)  # Day 11 observed target, held fixed.

prediction = weight * today + bias  # Scalar tensor 4.0.
loss = (prediction - tomorrow).square()  # Scalar tensor 1.0.
loss.backward()  # Store gradients 8 and 2 without updating the parameters.

# item() converts a one-element tensor to a Python number for display.
print("Prediction and loss", prediction.item(), loss.item())
print("Weight and bias gradients", weight.grad.item(), bias.grad.item())

Prediction and loss 4.0 1.0
Weight and bias gradients 8.0 2.0


**Try it.** Change the initial weight to 0.5 and rerun the cell. Before running, predict whether the gradient signs will change. Restore the weight to 1 afterwards.

<details>
<summary>Show solution</summary>

The prediction becomes 2 and the residual becomes −1. The gradients become −8 and −2. Increasing either parameter a little now helps because the prediction is too low.

</details>

## 3 · Use a gradient to update the model

**Gradient descent** subtracts the gradient multiplied by a **learning rate**. Use a learning rate of 0.05. The new weight is 1 − 0.05 × 8 = 0.6. The new bias is 0 − 0.05 × 2 = −0.1. The next forward pass predicts 2.3. It crosses the observed value, but the miss shrinks from 1.0 to 0.7. A larger step could cross further and make the loss increase.

The class below wraps `nn.Linear(1, 1)`. A **module** owns parameters and defines how inputs produce outputs in `forward`. The layer starts with a feature column of shape `(samples, 1)` and outputs a column of the same shape. The model removes the length-one output axis to match the target vector `(samples,)`.

An **optimizer** updates parameters. Here, SGD uses the gradient from the data passed to the step, multiplied by the learning rate. Our training function clears any old gradients, computes the loss, runs backpropagation, and then updates the parameters. Clearing matters because PyTorch adds new gradients to any gradients already stored.

There are three distinct actions in the example below. **Initialise** creates a model and sets its starting numbers. **Train** uses the known day 11 observation to calculate gradients and update those numbers. **Evaluate** runs a fresh prediction with the updated numbers. We evaluate on the same example here to check the arithmetic of one update. Section 4 switches to later days to check generalisation.

`with torch.no_grad():` temporarily pauses gradient recording inside the indented block. We use it here because setting the starting values is an assignment, outside the calculation we will differentiate. The weight and bias remain trainable. When the block ends, the training forward pass records operations as usual.

In [ ]:
import torch  # Provides tensors, no_grad, and SGD.

# Import the model class and the two functions that use it.
from pytorch_lab.nn.runoff_regression import RunoffRegression
from pytorch_lab.training import training_step, validation_step

# Construct a new model. This creates one linear layer and its trainable
# weight (shape (1, 1)) and bias (shape (1,)), with default initial values.
model = RunoffRegression()

# Pause gradient recording while we replace the defaults with the values
# from the hand calculation. Tracking resumes after this block.
with torch.no_grad():
    model.linear.weight.fill_(1.0)  # Set the existing weight to 1 in place.
    model.linear.bias.zero_()       # Set the existing bias to 0 in place.

# Double brackets give one row with one feature. The target has one axis
# because forward() returns one prediction per row as a vector.
features = torch.tensor([[4.0]])  # (1, 1), day 10 flow
target = torch.tensor([3.0])      # (1,), day 11 observed flow

# Give SGD references to the model's parameters and a learning rate.
# Creating the optimizer prepares it. The update happens inside training_step.
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

# Predict 4, compare it with 3, calculate gradients and update the parameters.
# The return is the pre-update loss as a Python float (1.0).
before = training_step(model, features, target, optimizer)

# Run a fresh forward pass using weight 0.6 and bias -0.1. Returns the
# prediction vector [2.3] and a scalar loss tensor (about 0.49).
after_prediction, after_loss = validation_step(model, features, target)

# Format numbers to two decimals. item() extracts a Python number from
# a tensor with one element, even when the tensor has shape (1,).
print(f"Loss before update {before:.2f}")
print(f"New prediction {after_prediction.item():.2f}, new loss {after_loss.item():.2f}")

**Try it.** Set the learning rate to 0.1 in the cell above. Calculate the new weight and bias before rerunning the whole cell. What happens to the loss?

<details>
<summary>Show solution</summary>

The weight becomes 0.2 and the bias becomes −0.2. The forecast becomes 0.6, so the squared error grows to 5.76. The gradient pointed in a useful local direction, but the step was too large.

</details>

## 4 · Fit on training targets and score on later targets

A **batch** is the set of examples used for one update. An **epoch** is one pass through the training examples. Here we use all ten training rows as one batch, so one epoch is one update.

Look closely at the last training example and first validation example. The training loss can use day 11's observed flow of 3 because we're fitting through the end of day 11. Once the model is fitted, it uses that same known value as the feature for day 12. You could run the forecast as `model(x_val[:1])` without passing day 12's actual flow. Once day 12 is observed, you can compute its error. The validation function only returns predictions and loss. You can inspect the function to see where the two paths differ.

| Call | What it reads | What you get back | What happens to the parameters |
| --- | --- | --- | --- |
| `training_step(model, x_train, y_train, optimizer)` | Ten features and their ten observed targets | One Python number for the loss before the update | SGD updates weight and bias |
| `model(x_val[:1])` inside `no_grad` | The day 11 flow of 3 m³/s | A forecast vector with one element | Weight and bias stay fixed |
| `validation_step(model, x_val, y_val)` | Three features and their three observed targets | A prediction vector and a scalar loss tensor | Weight and bias stay fixed |

The key difference is what we do with the error. In training, `loss.backward()` computes gradients and `optimizer.step()` uses them. In validation, the loss is still calculated because we have the observations by scoring time. No gradient or optimizer step is needed. Evaluation mode also tells layers such as dropout how to behave. Gradient recording is controlled separately by `no_grad`.

In [ ]:
import torch  # Provides tensors, no_grad, and SGD.

# Import the same components as Section 3 so this cell can run on its own.
from pytorch_lab.nn.runoff_regression import RunoffRegression
from pytorch_lab.training import training_step, validation_step

# Build one tensor with 17 synthetic daily mean flows in m³/s.
q = torch.tensor([2., 3., 7., 5., 3., 2., 4., 9., 6., 4., 3., 2., 4., 8., 8., 7., 6.])

# Pair each day's flow with the following day's observation.
# Unsqueeze inserts a feature axis so nn.Linear can read one feature per row.
x = q[:-1].unsqueeze(1)  # (17,) → (16,) → (16, 1), feature days 1–16
y = q[1:]               # (17,) → (16,), target days 2–17

# Split by target day. Row index 9 predicts day 11 from day 10.
# Row index 10 predicts day 12 using the already observed day 11 flow.
x_train, y_train = x[:10], y[:10]    # (10, 1), (10,), target days 2–11
x_val, y_val = x[10:13], y[10:13]    # (3, 1), (3,), target days 12–14

# Seed PyTorch's random number generator so rerunning this cell starts
# the new layer from the same initial weight and bias.
torch.manual_seed(7)
model = RunoffRegression()  # New model with its own parameters.
optimizer = torch.optim.SGD(model.parameters(), lr=0.02)  # Attach SGD to this model.

# range excludes 301, so this runs epochs 1 through 300. Each call uses
# all ten training samples and updates the parameters in place once.
for epoch in range(1, 301):
    train_loss = training_step(model, x_train, y_train, optimizer)
    # Print four snapshots of the pre-update MSE as training progresses.
    if epoch in (1, 20, 100, 300):
        print(f"Epoch {epoch:3d} training MSE before update {train_loss:.3f}")

# Issue the day 12 prediction from day 11's flow. Slicing preserves the
# sample and feature axes. no_grad avoids building a graph for this forecast.
with torch.no_grad():
    forecast_day_12 = model(x_val[:1])  # (1, 1) → (1,), about 4.07 m³/s

# Once observations are available, score the three validation forecasts.
# Discard the prediction vector with _ because we only need the MSE here.
_, val_loss = validation_step(model, x_val, y_val)  # loss shape ()

# y_val[0] selects the actual day 12 flow (2.0 m³/s). item() extracts each
# single value for printing. The validation MSE averages three errors squared.
print(f"Day 12 forecast {forecast_day_12.item():.2f} m³/s, observed {y_val[0].item():.2f} m³/s")
print(f"Validation MSE {val_loss.item():.3f}")

**Try it.** Open `pytorch_lab/training.py`. Which two lines perform backpropagation and change parameter values? Why can validation still compute a loss?

<details>
<summary>Show solution</summary>

`loss.backward()` computes gradients and `optimizer.step()` updates parameters. Validation shares the forward pass and MSE calculation through `common_step`. Its actual discharge values are needed for scoring. The `torch.no_grad()` decorator disables gradient tracking during that step.

</details>

## How the files connect to DROP

The notebook assembles data, model, and optimizer. `pytorch_lab/nn/runoff_regression.py` defines the forward calculation. `pytorch_lab/training.py` calls that model and shares the loss calculation between training and evaluation.

In the sibling repo, open `../drop-ch/project/nn/runoff_net.py` and find `RunoffNet.common_step`, `training_step`, `validation_step`, and `configure_optimizers`. Start by following the calls between those methods. The lab exposes the optimizer calls directly. DROP uses Lightning, whose trainer manages those calls around the model hooks. Later lessons will introduce the time-series data and model components those hooks depend on.